# Distributed XGBoost Training — Actuarial Pure Premium

This notebook trains a homeowners insurance pure premium model using **distributed XGBoost**
on a multi-node Ray cluster running inside Snowflake. With Snowflake ML, scaling from
single-node to distributed training is as simple as swapping `XGBRegressor` for
`XGBEstimator` and calling `scale_cluster()` — no Spark clusters, no Kubernetes YAML,
no infrastructure to manage.

## What is Ray?

[Ray](https://www.ray.io/) is an open-source framework for scaling Python workloads across
multiple machines. It handles task scheduling, data sharding, fault tolerance, and
inter-node communication — so you write single-machine logic and Ray parallelizes it.
Snowflake embeds Ray directly into its Container Runtime, giving you a managed distributed
compute layer without provisioning or configuring any infrastructure outside Snowflake.

## Ray on Snowflake — How It Works

Snowflake's Container Runtime includes a **pre-configured Ray cluster** that runs alongside
your notebook. Here's the architecture:

```
┌─────────────────────────────────────────────────────────────────┐
│  Snowflake Compute Pool (SPCS)                                  │
│                                                                 │
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐          │
│  │  Head Node   │  │  Worker 1    │  │  Worker 2    │          │
│  │  (Notebook)  │──│  (Ray)       │──│  (Ray)       │          │
│  │  + Ray Head  │  │              │  │              │          │
│  └──────────────┘  └──────────────┘  └──────────────┘          │
│         │                                                       │
│         ▼                                                       │
│  ┌─────────────────────────────────────┐                        │
│  │  Ray Dashboard (live monitoring)    │                        │
│  │  + Grafana metrics                  │                        │
│  └─────────────────────────────────────┘                        │
└─────────────────────────────────────────────────────────────────┘
         │
         ▼
┌─────────────────────────────────────────────────────────────────┐
│  Snowflake Storage                                              │
│  Tables, Stages, Feature Store, Model Registry                  │
└─────────────────────────────────────────────────────────────────┘
```

**Key concepts:**

- **`scale_cluster(N)`** — Dynamically adds worker nodes to the Ray cluster. The notebook
  itself runs on the head node; workers handle distributed computation.
- **`XGBEstimator`** — Snowflake's wrapper around Ray XGBoost. It handles data sharding,
  distributed communication, and fault tolerance automatically. You provide a
  `DataConnector` (pointer to a Snowflake table) and the estimator distributes training
  across all available workers.
- **`DataConnector`** — A lightweight reference to a Snowflake table that Ray workers read
  in parallel (each worker reads its own shard). No data is moved to the client.
- **Ray Dashboard** — A live web UI showing cluster utilization, active jobs, and per-node
  resource consumption. The URL is printed by `get_ray_dashboard_url()`.

**When to use distributed vs single-node:**

| Scenario | Use |
|----------|-----|
| Dataset fits in warehouse memory (<10GB) | `XGBRegressor` (single-node, warehouse) |
| Large dataset or long training time | `XGBEstimator` (distributed, compute pool) |
| Need GPU acceleration | `XGBEstimator` with `use_gpu=True` |

**Requirements:** This notebook must run on a Container Runtime service (not a warehouse).

---

## 1. Session Setup

Establish a Snowpark session and point it at the database/schema where our Feature Store
datasets and Model Registry live.

In [ ]:
# -- Session Setup --
# Connect to Snowflake and set the database/schema context.
# The Container Runtime provides get_active_session() automatically.
import sys
sys.path.append("..")

from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
from snowflake.ml.dataset import load_dataset
from src.config import DATABASE, SCHEMA, COMPUTE_POOL

# Get the pre-configured Snowpark session from Container Runtime
session = get_active_session()
session.sql(f"USE DATABASE {DATABASE}").collect()
session.sql(f"USE SCHEMA {SCHEMA}").collect()

print(f"Connected: {DATABASE}.{SCHEMA}")


## 2. Scale the Ray Cluster

`scale_cluster()` requests additional compute nodes from the pool. These nodes join
the Ray cluster as workers. The Ray Dashboard URL provides a live view of cluster
utilization, active jobs, memory pressure, and per-node resource allocation.

In [ ]:
# -- Scale the Ray Cluster --
# scale_cluster() adds worker nodes to the Ray cluster.
# The head node (this notebook) coordinates; workers execute distributed tasks.
# expected_cluster_size=3 means 1 head + 2 workers.
from snowflake.ml.runtime_cluster import scale_cluster
from snowflake.ml.runtime_cluster.cluster_manager import (
    get_cluster_size,
    get_ray_dashboard_url,
)

# Scale up: request 3 total nodes for distributed training
scale_cluster(expected_cluster_size=3)

# Print cluster info and the Ray Dashboard URL for live monitoring
print(f"Cluster size: {get_cluster_size()} nodes")
print(f"Ray Dashboard: {get_ray_dashboard_url()}")

## 3. Load Feature Store Datasets

Load pre-built training and validation datasets from the Snowflake Feature Store.
These contain engineered features (75 rating factors) joined onto policy spines
with a 75/25 train/validation split.

In [ ]:
# -- Load Feature Store Datasets --
# These versioned datasets were generated by the Feature Store in the main
# actuarial_pricing_demo notebook. They contain pre-engineered features
# joined onto policy spines with train/validation splits.
training_dataset = load_dataset(session, name=f"{DATABASE}.{SCHEMA}.ACTUARIAL_TRAINING", version="3")
validation_dataset = load_dataset(session, name=f"{DATABASE}.{SCHEMA}.ACTUARIAL_VALIDATION", version="3")

# Convert to Snowpark DataFrames for inspection and table materialization
train_sdf = training_dataset.read.to_snowpark_dataframe()
val_sdf = validation_dataset.read.to_snowpark_dataframe()

# Derive feature columns by excluding spine/label columns
_spine_cols = {"POLICY_ID", "PURE_PREMIUM", "EXPOSURE"}
feature_cols = [c for c in train_sdf.columns if c not in _spine_cols]

print(f"Features: {len(feature_cols)} | Train rows: {train_sdf.count()}")

## 4. Distributed XGBoost Training

`XGBEstimator` wraps Ray XGBoost for distributed gradient boosting. The data is
automatically sharded across workers via `DataConnector`. Each worker computes
gradients on its shard; the head node aggregates and updates the global model.

**Note:** `DataConnector.from_dataframe()` requires a single-query DataFrame.
Dataset reads produce multi-query plans, so we materialize to temp tables first.

In [ ]:
# -- Distributed XGBoost Training --
# XGBEstimator requires a DataConnector (pointer to a single-query table).
# Dataset DataFrames have complex query plans, so we materialize them into
# temporary tables first, then create DataConnectors from those tables.
from snowflake.ml.modeling.distributors.xgboost import XGBEstimator, XGBScalingConfig
from snowflake.ml.data.data_connector import DataConnector

# Materialize datasets as tables (DataConnector needs single-query sources)
train_sdf.write.mode("overwrite").save_as_table(f"{DATABASE}.{SCHEMA}.DISTRIBUTED_TRAIN_TEMP")
val_sdf.write.mode("overwrite").save_as_table(f"{DATABASE}.{SCHEMA}.DISTRIBUTED_VAL_TEMP")

# Create DataConnectors — Ray workers read shards from these tables in parallel
train_connector = DataConnector.from_dataframe(session.table(f"{DATABASE}.{SCHEMA}.DISTRIBUTED_TRAIN_TEMP"))
eval_connector = DataConnector.from_dataframe(session.table(f"{DATABASE}.{SCHEMA}.DISTRIBUTED_VAL_TEMP"))

# Configure the distributed XGBoost estimator
# - params: standard XGBoost hyperparameters
# - scaling_config: controls how Ray distributes work across the cluster
#   num_workers=-1 means auto-detect (use all available nodes)
#   num_cpu_per_worker=-1 means auto-detect (use all CPUs per node)
estimator = XGBEstimator(
    params={
        "objective": "reg:squarederror",  # Regression loss for pure premium
        "n_estimators": 200,              # Number of boosting rounds
        "learning_rate": 0.05,            # Step size shrinkage
        "max_leaves": 31,                 # Maximum leaf nodes per tree
    },
    scaling_config=XGBScalingConfig(
        num_workers=-1,          # Auto-detect: uses all cluster nodes
        num_cpu_per_worker=-1,   # Auto-detect: uses all CPUs per worker
    ),
)

# fit() distributes training across the Ray cluster.
# Each worker loads its shard of the data and participates in distributed
# gradient computation. Returns an xgboost.Booster object.
booster = estimator.fit(
    dataset=train_connector,       # Training data (sharded across workers)
    input_cols=feature_cols,       # Feature column names
    label_col="PURE_PREMIUM",      # Target column
    eval_set=eval_connector,       # Validation set for early stopping metrics
    verbose_eval=50,               # Print eval metrics every 50 rounds
)
print("Distributed training complete.")

In [ ]:
# -- Evaluate the Model --
# The distributed estimator returns a standard xgboost.Booster.
# We evaluate locally using DMatrix (the validation set is small enough).
import xgboost as xgb
import numpy as np

# Pull validation data to pandas for scoring
val_pdf = val_sdf.select(feature_cols + ["PURE_PREMIUM"]).to_pandas()

# Create XGBoost DMatrix for prediction
dval = xgb.DMatrix(val_pdf[feature_cols], label=val_pdf["PURE_PREMIUM"])

# Generate predictions and compute metrics
preds = booster.predict(dval)
rmse = float(np.sqrt(np.mean((val_pdf["PURE_PREMIUM"] - preds) ** 2)))
mae = float(np.mean(np.abs(val_pdf["PURE_PREMIUM"] - preds)))

print(f"Validation RMSE: {rmse:.4f}")
print(f"Validation MAE:  {mae:.4f}")

## 7. Scale Down

Release worker nodes back to the compute pool. You only pay for extra nodes while
actively training — always scale down when done.

In [ ]:
# -- Scale Down --
# Release worker nodes back to the compute pool when training is complete.
# This reduces cost — you only pay for extra nodes while actively training.
scale_cluster(expected_cluster_size=1)
print("Cluster scaled back to 1 node.")